# 01 — Python Data Structures

    - Choose the right built-in collection
- Reason about references, hashing, and complexity
- Practice interview patterns with AI-flavored data

    **Estimated study/practice time:** 3–4 hours  
    **Prerequisites:** Basic Python syntax

    **Priority legend:** 🔴 **Must Know** · 🟠 **Important** · 🟢 **Good to Know**

    ## Table of contents

    1. [Lists](#lists)
2. [References and copying](#references-and-copying)
3. [Dictionaries and hashing](#dictionaries-and-hashing)
4. [Sets](#sets)
5. [Tuples and mutability](#tuples-and-mutability)
6. [Selection drills](#selection-drills)
7. [Practice](#practice)

    > Active-practice rule: run examples, predict output before revealing it, complete every `TODO`, then compare with the solution below it.


## Environment setup

```bash
conda create -n python-ai-interview python=3.11 -y
conda activate python-ai-interview
pip install jupyter numpy pandas matplotlib ipykernel
python -m ipykernel install --user --name python-ai-interview --display-name "Python AI Interview"
```

Registering a kernel lets Jupyter and VS Code select this exact environment and its installed packages.


## Lists

🔴 **Must Know**

**Concept.** A list is an ordered, mutable dynamic array.  
**Why it matters.** Lists commonly hold batches, tokens, predictions, and records.  
**Syntax.** `nums = [1, 2, 3]`; indexing `nums[0]`, negative indexing `nums[-1]`, slicing `nums[start:end:step]`.  
**How Python executes it.** A list stores references in a contiguous internal array and occasionally reallocates extra capacity, making `append` amortized O(1).

**Syntax:** `nums = [1, 2, 3]`; indexing `nums[0]`, negative indexing `nums[-1]`, slicing `nums[start:end:step]`.

| Operation | Typical time |
|---|---:|
| index | O(1) |
| append / pop end | amortized O(1) / O(1) |
| insert beginning / remove by value | O(n) |
| membership | O(n) |
| sort | O(n log n) |

**Interview question:** Why is inserting at index 0 slow? Every existing reference must shift right.
**Exercise/solution:** complete *slice batches* in the Practice section before viewing its solution.


In [ ]:
nums = [3, 1, 2]
print(nums[0], nums[-1], nums[:3], nums[2:], nums[::-1], nums[::2])
nums.append([4, 5])      # one nested element
print(nums)
nums = [3, 1, 2]
nums.extend([4, 5])      # two elements
nums.insert(0, 9)
nums.remove(3)
last = nums.pop()
sorted_copy = sorted(nums)  # new list
nums.sort()                  # mutates, returns None
nums.reverse()
print(nums, sorted_copy, last, 2 in nums)


**Common interview mistake:** assigning `result = nums.sort()` makes `result` be `None`. Slice end is exclusive; `nums[:3]` contains indices 0–2.

### List slicing

**Concept.** `nums[start:end:step]` creates a shallow list copy of the selected references.  
**Why it matters.** Slicing is common in batching, windows, and reversing interview inputs.  
**Syntax/example.** `nums[:3]`, `nums[2:]`, `nums[::-1]`, and `nums[::2]`.  
**How Python executes it.** Python normalizes the indices, walks the selected positions, and copies their references into a new outer list.  
**Complexity.** Time and space are O(k) for k selected elements.  
**Common interview mistake.** Treating the end index as inclusive.  
**Interview question.** Does a list slice deep-copy nested values? No.  
**Exercise/solution.** Complete *slice batches* below.

## References and copying

🔴 **Must Know**

**Concept.** Assignment copies a reference, not an object. A shallow copy (`copy()` or `[:]`) creates a new outer list but shares nested objects; `deepcopy` recursively copies the reachable structure.  
**Why it matters.** AI preprocessing should not mutate caller-owned batches accidentally.  
**Syntax.** `alias = a`, `shallow = a.copy()`, `shallow = a[:]`, `deep = copy.deepcopy(a)`.  
**How Python executes it.** Names point to objects; shallow copying duplicates only the outer reference array.  
**Complexity.** Shallow list copy is O(n); deep-copy cost is proportional to the reachable object graph.  
**Common interview mistake.** Assuming a shallow copy makes nested lists independent.  
**Interview question.** Why do changes through `b = a` appear through `a`? Both names reference one object.  
**Exercise/solution.** Complete *safe metadata* below.

```text
a ──┐
    ├──> [1, 2, 3]
b ──┘

shallow_a --> [ inner ──> [1, 2] ]
shallow_b --> [ inner ──> [1, 2] ]
```


In [ ]:
import copy

a = [[1, 2], [3, 4]]
alias = a
shallow_1 = a.copy()
shallow_2 = a[:]
deep = copy.deepcopy(a)
alias.append([5])
shallow_1[0].append(99)
print("original", a)
print("same outer?", alias is a, shallow_1 is a)
print("shared inner?", shallow_1[0] is a[0], shallow_2[0] is a[0])
print("deep independent?", deep[0] is not a[0], deep)


## Dictionaries and hashing

🔴 **Must Know**

**Concept.** A dictionary maps unique hashable keys to values using a hash table.  
**Why it matters.** Dictionaries power counting, caching, grouping, ID lookup, and complement lookup in Two Sum.  
**Syntax.** `d[key] = value`, `d.get(key, default)`, `d.keys()`, `d.values()`, `d.items()`.  
**How Python executes it.** Conceptually:

```text
key → hash(key) → bucket/location → key equality check → value
```

**Complexity.** Lookup, insert, and delete are average O(1), worst-case O(n).  
**Common interview mistake.** `d[key]` raises `KeyError`; `d.get(key, default)` supplies a fallback.  
**Interview question.** Why must keys be hashable? Their hash/equality must remain stable while stored.  
**Exercise/solution.** Complete *group by source* and *first duplicate* below.


In [ ]:
user = {"name": "Alice", "age": 30}
user["role"] = "ML Engineer"
print(user["name"], user.get("country", "unknown"))
user.setdefault("skills", []).append("Python")
print("keys", list(user.keys()))
print("values", list(user.values()))
for key, value in user.items():
    print(key, value)
del user["age"]

tokens = ["rag", "llm", "rag"]
counts = {}
for token in tokens:
    counts[token] = counts.get(token, 0) + 1
from collections import Counter
print(counts, Counter(tokens))


**How Python executes it.** Python hashes the key, probes for a matching slot, then checks equality. Mutable lists cannot be keys because changing them would invalidate their location. A tuple works only when all its contents are hashable.

**Common mistake:** using `if d.get(key):` to test existence fails when the stored value is false-like; use `if key in d:`.

## Sets

🔴 **Must Know**

**Concept.** A set is a hash table of unique hashable values.  
**Why it matters.** It turns repeated membership checks and duplicate detection from O(n) per list lookup into average O(1).  
**Syntax.** `seen=set()`, `add`, `remove`, `discard`, union `|`, intersection `&`, difference `-`.  
**How Python executes it.** Values are hashed into table locations; collisions are resolved and equality confirms a match.  
**Complexity.** Average add, remove, and membership O(1); worst case O(n).  
**Common interview mistake.** `remove` raises if absent; `discard` does not.  
**Interview question.** Why is a set common in Contains Duplicate? It stores prior values with average O(1) membership.  
**Exercise/solution.** Complete *first duplicate* and *deduplicate* below.


In [ ]:
seen = set()
for model_id in [10, 11, 10]:
    if model_id in seen:
        print("duplicate", model_id)
    seen.add(model_id)
seen.discard(999)
print({1, 2} | {2, 3}, {1, 2} & {2, 3}, {1, 2} - {2, 3})


## Tuples and mutability

🔴 **Must Know**

**Concept.** Tuples are ordered and immutable.  
**Why it matters.** They support unpacking, multiple return values, and—when all members are hashable—dictionary keys.  
**Syntax.** `point=(3,5)`, `x,y=point`, `return minimum, maximum`.  
**How Python executes it.** Tuple membership references cannot be replaced; hashability still depends on every contained object.  
**Complexity.** Indexing O(1); membership O(n); construction O(n).  
**Common interview mistake.** Assuming every tuple is hashable even when it contains a list.  
**Interview question.** Why use `tuple(sorted(word))`? It converts the sorted list into a stable Group Anagrams key.  
**Exercise/solution.** Complete *matrix coordinates* below.

| Type | Mutable? | Normally hashable? |
|---|---|---|
| list | Yes | No |
| dict | Yes | No |
| set | Yes | No |
| tuple | No | If contents are hashable |
| str / int | No | Yes |


In [ ]:
point = (3, 5)
x, y = point
def min_and_max(values):
    return min(values), max(values)
minimum, maximum = min_and_max([3, 1, 5])
anagram_key = tuple(sorted("listen"))
grouped = {anagram_key: ["listen", "silent"]}
print(x, y, minimum, maximum, grouped)


## Selection drills

Choose before revealing: 1 fast membership—**set**; 2 key→value—**dict**; 3 ordered mutable sequence—**list**; 4 immutable composite key—**tuple**; 5 FIFO—**deque**; 6 LIFO—**list**; 7 count labels—**Counter/dict**; 8 group documents—**defaultdict(list)**; 9 deduplicate IDs—**set**; 10 preserve duplicates and order—**list**; 11 cache by parameters—**dict**; 12 coordinate—**tuple**; 13 BFS frontier—**deque**; 14 priority frontier—**heap**; 15 top-level JSON payload—**dict**.


## Practice


### Exercise — 🟢 Easy: deduplicate

        **Goal:** Return unique values while preserving order.

        **Example**

        ```text
        Input: [3,1,3,2]
Output: [3,1,2]
        ```

        <details><summary>Hint 1</summary>Track first occurrences.</details>
        <details><summary>Hint 2</summary>Dictionary keys preserve insertion order.</details>


In [ ]:
def deduplicate(items):
    # TODO: Write your solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def deduplicate(items):
    return list(dict.fromkeys(items))

assert deduplicate([3, 1, 3, 2]) == [3, 1, 2]


**Complexity**

```text
Time: O(n) average; Space: O(n)
```

**Common mistake:** Using `set(items)` loses ordering.


### Exercise — 🟢 Easy: first duplicate

        **Goal:** Return the first value seen twice, or None.

        **Example**

        ```text
        Input: [2,1,3,2]
Output: 2
        ```

        <details><summary>Hint 1</summary>Use a set.</details>
        <details><summary>Hint 2</summary>Check before adding.</details>


In [ ]:
def first_duplicate(items):
    # TODO: Write your solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def first_duplicate(items):
    seen = set()
    for item in items:
        if item in seen: return item
        seen.add(item)
    return None

assert first_duplicate([2,1,3,2]) == 2


**Complexity**

```text
Time: O(n) average; Space: O(n)
```

**Common mistake:** Adding before checking makes every item appear duplicate.


### Exercise — 🟢 Easy: group by source

        **Goal:** Group document dictionaries by `source`.

        **Example**

        ```text
        Input: two source=a docs
Output: {'a': [doc, doc]}
        ```

        <details><summary>Hint 1</summary>Map source to a list.</details>
        <details><summary>Hint 2</summary>`defaultdict(list)` removes initialization branches.</details>


In [ ]:
def group_by_source(docs):
    # TODO: Write your solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
from collections import defaultdict
def group_by_source(docs):
    groups = defaultdict(list)
    for doc in docs: groups[doc['source']].append(doc)
    return dict(groups)

assert len(group_by_source([{'source':'a'}, {'source':'a'}])['a']) == 2


**Complexity**

```text
Time: O(n); Space: O(n)
```

**Common mistake:** Using one shared list for every key.


### Exercise — 🟢 Easy: safe metadata

        **Goal:** Return a copied metadata dict with a score.

        **Example**

        ```text
        Input: {'page':1}, .8
Output: new dict
        ```

        <details><summary>Hint 1</summary>Copy before mutation.</details>
        <details><summary>Hint 2</summary>A shallow copy is enough for a top-level scalar.</details>


In [ ]:
def add_score(metadata, score):
    # TODO: Write your solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def add_score(metadata, score):
    result = metadata.copy()
    result['score'] = score
    return result

source = {'page': 1}
assert add_score(source, .8) == {'page':1, 'score':.8} and 'score' not in source


**Complexity**

```text
Time: O(n); Space: O(n)
```

**Common mistake:** Mutating caller-owned metadata.


### Exercise — 🟢 Easy: matrix coordinates

        **Goal:** Count repeated `(row, column)` coordinates.

        **Example**

        ```text
        Input: [(0,1),(0,1)]
Output: {(0,1):2}
        ```

        <details><summary>Hint 1</summary>Tuples are hashable.</details>
        <details><summary>Hint 2</summary>Use Counter.</details>


In [ ]:
def coordinate_counts(coords):
    # TODO: Write your solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
from collections import Counter
def coordinate_counts(coords):
    return Counter(coords)

assert coordinate_counts([(0,1),(0,1)])[(0,1)] == 2


**Complexity**

```text
Time: O(n); Space: O(k)
```

**Common mistake:** Trying to use lists as keys.


### Exercise — 🟢 Easy: slice batches

        **Goal:** Split a list into fixed-size batches.

        **Example**

        ```text
        Input: [1,2,3,4,5], 2
Output: [[1,2],[3,4],[5]]
        ```

        <details><summary>Hint 1</summary>Step by `size`.</details>
        <details><summary>Hint 2</summary>The last slice may be short.</details>


In [ ]:
def batches(items, size):
    # TODO: Write your solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def batches(items, size):
    if size <= 0: raise ValueError('size must be positive')
    return [items[i:i+size] for i in range(0, len(items), size)]

assert batches([1,2,3,4,5], 2) == [[1,2],[3,4],[5]]


**Complexity**

```text
Time: O(n); Space: O(n)
```

**Common mistake:** Forgetting to reject size zero.


## Key takeaways

    - Lists are dynamic arrays.
- Hash tables drive fast lookup.
- Assignment aliases; copying controls shared mutation.

    ## Mistakes to avoid

    - Using a list for repeated membership.
- Confusing shallow with deep copying.
- Using unhashable keys.

    ## 10 rapid-fire interview questions

    1. **Which collection gives average O(1) membership?** <details><summary>Answer</summary>A set.</details>
2. **Which is immutable: list or tuple?** <details><summary>Answer</summary>Tuple.</details>
3. **What does `dict.get` help avoid?** <details><summary>Answer</summary>`KeyError`, while supplying a default.</details>
4. **Does `sorted` mutate its input?** <details><summary>Answer</summary>No; it returns a new list.</details>
5. **Does `list.sort` return the sorted list?** <details><summary>Answer</summary>No; it mutates and returns `None`.</details>
6. **`append([2,3])` adds how many elements?** <details><summary>Answer</summary>One: the list object itself.</details>
7. **What does `extend([2,3])` do?** <details><summary>Answer</summary>Adds two elements by iterating the argument.</details>
8. **Typical list membership complexity?** <details><summary>Answer</summary>O(n).</details>
9. **Typical set membership complexity?** <details><summary>Answer</summary>Average O(1).</details>
10. **Typical list append complexity?** <details><summary>Answer</summary>Amortized O(1).</details>

    ## Readiness checklist

    - [ ] I can choose list/dict/set/tuple.
- [ ] I can explain aliasing.
- [ ] I can state typical operation costs.
